# Learning quantum channels on 4×4 sudoku

This notebook trains actual Optyx quantum channels on the recurrent geometry proposed in [optyx#13](https://github.com/rel-int/optyx/issues/13). Sixteen cell channels exchange qubit messages with four row, four column and four square channels. Their three-step unrolling is contracted with Cotengra on PyTorch tensors, and the sudoku loss is differentiated through the contraction. There is no classical neural-network interpreter.

The documentation run is deliberately small. It learns to rank the unique completion of a puzzle above a clue-preserving invalid completion, using twelve training and four held-out puzzles. This tests the quantum channel, recurrent routing, approximate contraction and automatic differentiation together; it is not presented as a full 288-puzzle solver benchmark. PyTorch is required only for this learning experiment, not by the Optyx core package.

In [1]:
import time

import numpy as np
import torch
from cotengra import ReusableHyperCompressedOptimizer
from discopy import tensor

from optyx.channel import Channel, Diagram, qubit
from optyx.core.backends import DiscopyBackend
from optyx.core.contract import contract_tensor
from optyx.core.diagram import Box as CoreBox, bit as core_bit
from optyx.interaction import Box, CMap
from optyx.qubits import Ket

torch.set_num_threads(1)
_ = torch.manual_seed(7)

## Trainable Optyx channels

Every local map is an eight-qubit unitary Kraus map inside `optyx.channel.Channel`. A product of real rotations appears on either side of sixteen trainable controlled rotations joining the first four and last four ports. One parameter tensor is shared by all cell boxes and another by all constraint boxes. Real orthogonal unitaries keep the singular-value gradient of compressed contraction well-defined, and the resulting dense local arrays remain connected to the PyTorch computation graph.

In [2]:
size = 4
n_cells = size ** 2
n_constraints = 3 * size
width = 8


def kron_all(arrays):
    result = arrays[0]
    for array in arrays[1:]:
        result = torch.kron(result, array)
    return result


def rotation_layer(parameters):
    matrices = []
    for theta in parameters:
        matrices.append(torch.stack((
            torch.stack((torch.cos(theta), -torch.sin(theta))),
            torch.stack((torch.sin(theta), torch.cos(theta))),
        )).to(torch.float64))
    return kron_all(matrices)


def channel_unitary(parameters):
    result = rotation_layer(parameters[:width])
    basis = torch.arange(2 ** width)
    interactions = parameters[2 * width:].reshape(4, 4)
    for control in range(4):
        for target in range(4):
            theta = interactions[control, target]
            selected = (((basis >> control) & 1) == 1) & (
                ((basis >> (4 + target)) & 1) == 0)
            lower = basis[selected]
            upper = lower ^ (1 << (4 + target))
            updated = result.clone()
            updated[lower] = (
                torch.cos(theta) * result[lower]
                - torch.sin(theta) * result[upper])
            updated[upper] = (
                torch.sin(theta) * result[lower]
                + torch.cos(theta) * result[upper])
            result = updated
    return rotation_layer(parameters[width:2 * width]) @ result


def trainable_channel(name, parameters):
    kraus = CoreBox(
        name, core_bit ** width, core_bit ** width,
        array=channel_unitary(parameters))
    return Channel(name, kraus, qubit ** width, qubit ** width)


cell_parameters = (
    torch.randn(4 * width, dtype=torch.float64) * .1
).requires_grad_()
constraint_parameters = (
    torch.randn(4 * width, dtype=torch.float64) * .1
).requires_grad_()

## The recurrent sudoku map

A cell reads three message qubits and writes three message qubits plus two prediction qubits. A constraint reads and writes four message qubits. The separate directions give 96 edges and 192 recurrent memory wires; the 32 prediction wires are the boundary.

In [3]:
def memberships(row, column):
    square = 2 * (row // 2) + column // 2
    position = 2 * (row % 2) + column % 2
    return (
        (n_cells + row, column),
        (n_cells + size + column, row),
        (n_cells + 2 * size + square, position),
    )


def sudoku_cmap():
    cell_channel = trainable_channel(
        "Cell", cell_parameters)
    constraint_channel = trainable_channel(
        "Constraint", constraint_parameters)
    boxes = [
        Box(f"cell_{index}", qubit ** 3, qubit ** 5, cell_channel)
        for index in range(n_cells)
    ]
    boxes += [
        Box(f"{kind}_{index}", qubit ** 4, qubit ** 4,
            constraint_channel)
        for kind in ("row", "column", "square")
        for index in range(size)
    ]
    edges = []
    for cell in range(n_cells):
        row, column = divmod(cell, size)
        for slot, (constraint, position) in enumerate(
                memberships(row, column)):
            edges += [
                ((cell, slot), (constraint, size + position)),
                ((cell, 3 + slot), (constraint, position)),
            ]
    return CMap(boxes, edges)


sudoku = sudoku_cmap()
topology = {
    "boxes": len(sudoku.boxes),
    "edges": len(sudoku.edges),
    "prediction_wires": len(sudoku.boundary),
    "memory_wires": len(sudoku.memory),
}
assert topology == {
    "boxes": 28, "edges": 96,
    "prediction_wires": 32, "memory_wires": 192}
topology

{'boxes': 28, 'edges': 96, 'prediction_wires': 32, 'memory_wires': 192}

## A balanced dataset

We enumerate all 288 completed grids. Each selected puzzle hides cells 0 and 1 plus six others, and its eight remaining clues uniquely identify one grid in the complete corpus. The negative completion swaps the two hidden values in the first row, so it preserves every clue but violates sudoku constraints. Twelve solutions are used for training and four disjoint solutions for evaluation.

In [4]:
def sudoku_groups():
    rows = [tuple(row * size + column for column in range(size))
            for row in range(size)]
    columns = [tuple(row * size + column for row in range(size))
               for column in range(size)]
    squares = [
        tuple((2 * block_row + row) * size
              + 2 * block_column + column
              for row in range(2) for column in range(2))
        for block_row in range(2) for block_column in range(2)
    ]
    return rows + columns + squares


groups = sudoku_groups()
peers = [set() for _ in range(n_cells)]
for group in groups:
    for cell in group:
        peers[cell].update(set(group) - {cell})


def enumerate_solutions():
    result, grid = [], [0] * n_cells

    def visit(cell):
        if cell == n_cells:
            result.append(np.array(grid))
            return
        for value in range(1, size + 1):
            if all(grid[peer] != value for peer in peers[cell]):
                grid[cell] = value
                visit(cell + 1)
        grid[cell] = 0

    visit(0)
    return np.stack(result)


solutions = enumerate_solutions()
assert solutions.shape == (288, n_cells)

In [5]:
def puzzle_case(solution, random):
    available = np.arange(2, n_cells)
    while True:
        clue_cells = random.choice(
            available, n_cells // 2, replace=False)
        matches = np.all(
            solutions[:, clue_cells] == solution[clue_cells], axis=1)
        if matches.sum() == 1:
            clue = np.zeros(n_cells, dtype=int)
            clue[clue_cells] = solution[clue_cells]
            invalid = solution.copy()
            invalid[0], invalid[1] = invalid[1], invalid[0]
            return clue, solution, invalid


random = np.random.default_rng(19)
selection = random.choice(len(solutions), 16, replace=False)
cases = [puzzle_case(solutions[index], random) for index in selection]
train_cases, test_cases = cases[:12], cases[12:]
dataset = {
    "complete_corpus": len(solutions),
    "training_puzzles": len(train_cases),
    "held_out_puzzles": len(test_cases),
    "clues_per_puzzle": int(np.count_nonzero(train_cases[0][0])),
}
dataset

{'complete_corpus': 288,
 'training_puzzles': 12,
 'held_out_puzzles': 4,
 'clues_per_puzzle': 8}

## Efficient tensor semantics of the unrolling

Materialising a permutation on 224 qubit wires obscures the useful tensor-network structure. `unrolled_tensor_map` constructs the equivalent `discopy.tensor.CMap` directly: at time zero every paired port reads an initial memory state, at later times it reads the previous output of its `CMap.partner`, and the final outputs meet the candidate effect. Thus swaps become index labels rather than thousands of tensor boxes. The local tensors are the Kraus arrays of the Optyx channels above.

In [6]:
zero = torch.tensor([1, 0], dtype=torch.float64)
one = torch.tensor([0, 1], dtype=torch.float64)
plus = torch.tensor([1, 1], dtype=torch.float64) / 2 ** .5


def digit_vectors(values):
    vectors = []
    for value in values:
        if value == 0:
            vectors += [plus, plus]
        else:
            value -= 1
            vectors += [
                one if value // 2 else zero,
                one if value % 2 else zero,
            ]
    return vectors


def unrolled_tensor_map(cmap, clues, completion, n_steps):
    boxes, pairs, next_port = [], [], 0

    def add(box):
        nonlocal next_port
        base = next_port
        next_port += len(box.dom) + len(box.cod)
        boxes.append(box)
        dom = list(range(base, base + len(box.dom)))
        cod = list(reversed(range(base + len(box.dom), next_port)))
        return dom, cod

    def state(value):
        box = tensor.Box(
            "state", tensor.Dim(), tensor.Dim(2), value)
        return add(box)[1][0]

    def effect(value):
        box = tensor.Box(
            "effect", tensor.Dim(2), tensor.Dim(), value)
        return add(box)[0][0]

    boundary, ports = cmap.boundary, cmap.ports
    clue_vectors = digit_vectors(clues)
    boundary_inputs = [
        [state(value) for value in clue_vectors]
        for _ in range(n_steps)
    ]
    memory_inputs = {
        port: state(plus) for port in ports if port in cmap.partner
    }
    with tensor.backend("pytorch"):
        local_tensors = {
            id(box.channel): tensor.Box(
                box.channel.name, tensor.Dim(2) ** width,
                tensor.Dim(2) ** width, box.channel.kraus.array)
            for box in cmap.boxes
        }
    local_inputs, local_outputs = [], []
    for _ in range(n_steps):
        step_inputs, step_outputs = {}, {}
        for box_index, box in enumerate(cmap.boxes):
            dom, cod = add(local_tensors[id(box.channel)])
            for port_index in range(len(box.ports)):
                port = box_index, port_index
                step_inputs[port] = dom[port_index]
                step_outputs[port] = cod[port_index]
        local_inputs.append(step_inputs)
        local_outputs.append(step_outputs)
    for step in range(n_steps):
        for boundary_index, port in enumerate(boundary):
            pairs.append((
                boundary_inputs[step][boundary_index],
                local_inputs[step][port]))
        for port in ports:
            if port not in cmap.partner:
                continue
            source = memory_inputs[port] if step == 0 else \
                local_outputs[step - 1][cmap.partner[port]]
            pairs.append((source, local_inputs[step][port]))
    completion_vectors = digit_vectors(completion)
    for step in range(n_steps):
        vectors = [plus] * len(boundary) \
            if step < n_steps - 1 else completion_vectors
        for port, value in zip(boundary, vectors):
            pairs.append((local_outputs[step][port], effect(value)))
    for port in ports:
        if port in cmap.partner:
            pairs.append((
                local_outputs[-1][cmap.partner[port]], effect(plus)))
    edges = list(range(next_port))
    for left, right in pairs:
        edges[left], edges[right] = right, left
    assert all(edge != index for index, edge in enumerate(edges))
    return tensor.CMap(
        tensor.Dim(), tensor.Dim(), tuple(boxes), edges)

In [7]:
n_steps = 3
example_network = unrolled_tensor_map(
    sudoku, train_cases[0][0], train_cases[0][1], n_steps)
network_summary = {
    "steps": n_steps,
    "tensor_boxes": len(example_network.boxes),
    "tensor_ports": len(example_network.ports),
}
assert network_summary == {
    "steps": 3, "tensor_boxes": 660, "tensor_ports": 1920}
network_summary

{'steps': 3, 'tensor_boxes': 660, 'tensor_ports': 1920}

## Born loss, contraction and backpropagation

The scalar contraction is a postselected amplitude for a candidate completion. Its squared magnitude is a Born score. Cross entropy normalises the correct and invalid scores for each puzzle. Cotengra reuses one compressed contraction path with bond dimension four, while PyTorch records every contraction and singular-value operation for reverse-mode differentiation.

In [8]:
path_optimizer = ReusableHyperCompressedOptimizer(
    chi=4, methods=("greedy-compressed",),
    max_repeats=1, progbar=False, parallel=False)


def completion_energy(clues, completion):
    network = unrolled_tensor_map(
        sudoku_cmap(), clues, completion, n_steps)
    amplitude = contract_tensor(
        network, backend="pytorch", optimize=path_optimizer,
        max_bond=4, cutoff=1e-8, dtype=float).array
    probability = amplitude.abs().square().real
    return -torch.log(probability.clamp_min(
        torch.finfo(torch.float64).tiny))


def correct_probabilities(dataset):
    probabilities = []
    for clues, solution, invalid in dataset:
        energies = torch.stack((
            completion_energy(clues, solution),
            completion_energy(clues, invalid),
        ))
        probabilities.append(torch.softmax(-energies, 0)[0].item())
    return probabilities


initial_test = correct_probabilities(test_cases)
initial_test

[0.4959578078757143,
 0.5032892498318593,
 0.47536503660952467,
 0.5322588706658481]

In [9]:
learning_optimizer = torch.optim.Adam(
    (cell_parameters, constraint_parameters), lr=0.03)
history = []
gradient_norm = None
started = time.perf_counter()
for step in range(24):
    clues, solution, invalid = train_cases[step % len(train_cases)]
    learning_optimizer.zero_grad()
    energies = torch.stack((
        completion_energy(clues, solution),
        completion_energy(clues, invalid),
    ))
    loss = energies[0] + torch.logsumexp(-energies, 0)
    loss.backward()
    if step == 0:
        gradient_norm = torch.sqrt(sum(
            parameter.grad.square().sum()
            for parameter in (
                cell_parameters, constraint_parameters))).item()
    learning_optimizer.step()
    history.append(loss.item())

final_test = correct_probabilities(test_cases)
elapsed = time.perf_counter() - started
results = {
    "initial_gradient_norm": gradient_norm,
    "first_epoch_loss": float(np.mean(history[:12])),
    "second_epoch_loss": float(np.mean(history[12:])),
    "initial_test_probability": float(np.mean(initial_test)),
    "final_test_probability": float(np.mean(final_test)),
    "initial_candidate_accuracy": float(
        np.mean(np.asarray(initial_test) > .5)),
    "final_candidate_accuracy": float(
        np.mean(np.asarray(final_test) > .5)),
    "training_seconds": elapsed,
}
assert results["initial_gradient_norm"] > 1e-6
assert results["second_epoch_loss"] < results["first_epoch_loss"]
assert (results["final_test_probability"]
        > results["initial_test_probability"])
results

{'initial_gradient_norm': 0.9782993003016354,
 'first_epoch_loss': 0.7048416443508886,
 'second_epoch_loss': 0.6632507943010696,
 'initial_test_probability': 0.5017177412457365,
 'final_test_probability': 0.5774273792551208,
 'initial_candidate_accuracy': 0.5,
 'final_candidate_accuracy': 0.75,
 'training_seconds': 254.63967854098883}

In [10]:
example = {
    "clues": test_cases[0][0].reshape(size, size).tolist(),
    "correct": test_cases[0][1].reshape(size, size).tolist(),
    "invalid": test_cases[0][2].reshape(size, size).tolist(),
    "correct_probability": final_test[0],
}
example

{'clues': [[0, 0, 0, 0], [0, 1, 0, 3], [4, 3, 2, 1], [0, 2, 0, 4]],
 'correct': [[3, 4, 1, 2], [2, 1, 4, 3], [4, 3, 2, 1], [1, 2, 3, 4]],
 'invalid': [[4, 3, 1, 2], [2, 1, 4, 3], [4, 3, 2, 1], [1, 2, 3, 4]],
 'correct_probability': 0.3464235579543045}

## Approximate fixpoint semantics

Three rounds are the finite stream semantics: a clue can travel from a cell to a constraint and back to a prediction. `CMap.fix` is the approximate stationary alternative and delegates depth and bond refinement to the same backend boundary. A dense fixed point of the complete 192-memory-qubit sudoku is not attempted in documentation, so the executable probe checks the API on one recurrent qubit.

In [11]:
wire = Box("wire", qubit, qubit ** 2, Diagram.id(qubit ** 3))
probe = CMap([wire], [((0, 1), (0, 2))])
fixed = probe.fix(
    Ket(1), Ket(0) @ Ket(0), n_steps=2,
    backend=DiscopyBackend())
assert np.allclose(fixed.density_matrix, [[0, 0], [0, 1]])

This experiment establishes the end-to-end path that was missing: Optyx channel parameters → recurrent quantum tensor map → Cotengra compression → PyTorch loss → channel gradients. Scaling the candidate set, batching contractions and comparing several bond dimensions remain benchmark work rather than hidden assumptions in this documentation run.